# Krisp Jobs Scraper

**Source:** [krisp.ai/careers](https://krisp.ai/careers/)  
**Rendering:** Server-side HTML (WordPress) — no JavaScript required  
**Filter:** Armenia-based positions only

### Scraping strategy
1. Fetch listing page — job cards with title, location, work type are all in HTML (`#job_listings` section)
2. Filter to Armenia jobs from the listing (no need to hit detail pages for filtering)
3. Fetch each Armenia job's detail page — content is in `div.job_data_container`

### Page structure
- Listing page: `#job_listings` → `<a href="/jobs/{slug}/">` with title + location + work_type as text nodes
- Detail page: `div.job_data_container` contains all structured text (no JS needed)

### Ethics & robots.txt
- `robots.txt` allows all crawling except `/wp-admin/` and tracking query params (checked 2026-03-22)
- Rate-limited: 1.5 s between detail page requests

### Output files
- `data/raw/jobs/krisp_jobs_raw.csv`
- `data/processed/jobs/krisp_jobs_standardized.csv`

In [1]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

CAREERS_URL = "https://krisp.ai/careers/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ThesisResearch/1.0; Armenian IT curriculum alignment; academic use)",
    "Accept-Language": "en-US,en;q=0.9",
}
DELAY_S = 1.5

BASE_DIR = Path.cwd()
RAW_DIR  = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR = BASE_DIR / "data" / "processed" / "jobs"

print(f"Target: {CAREERS_URL}")
print(f"Raw output:  {RAW_DIR}")
print(f"Proc output: {PROC_DIR}")

Target: https://krisp.ai/careers/
Raw output:  /Users/lianaaghamalyan/thesis_data/data/raw/jobs
Proc output: /Users/lianaaghamalyan/thesis_data/data/processed/jobs


## Helper functions

In [2]:
def html_to_text(html_str):
    """Strip HTML tags and normalize whitespace."""
    if not html_str:
        return ""
    text = BeautifulSoup(str(html_str), "html.parser").get_text("\n", strip=True)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

print("Helpers defined.")

Helpers defined.


## Step 1 — Collect job links from listing page

In [3]:
resp = requests.get(CAREERS_URL, headers=HEADERS, timeout=20)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")
listing = soup.find(id="job_listings")

job_entries = []
for a in listing.find_all("a", href=True):
    href = a["href"]
    if "/jobs/" not in href:
        continue
    url = href if href.startswith("http") else "https://krisp.ai" + href
    texts = [t.strip() for t in a.get_text("\n").split("\n") if t.strip()]
    job_entries.append({
        "title":     texts[0] if len(texts) > 0 else "",
        "location":  texts[1] if len(texts) > 1 else "",
        "work_type": texts[2] if len(texts) > 2 else "",
        "url":       url,
    })

print(f"Total jobs on careers page: {len(job_entries)}")
armenia = [j for j in job_entries if "armenia" in j["location"].lower()]
print(f"Armenia jobs: {len(armenia)}")
print()
for j in armenia:
    print(f"  {j['title']:55s} | {j['location']} | {j['work_type']}")

Total jobs on careers page: 11
Armenia jobs: 7

  Technical Support Engineering Manager, AI Meeting Assistant | Armenia | Hybrid
  Senior Customer Engineer, Enterprise                    | Armenia | Hybrid
  AI QA Engineer                                          | Armenia | Hybrid
  Lead ML Systems Engineer, Voice AI                      | Armenia | Hybrid
  ML Engineer, Voice AI                                   | Armenia | On-site
  Senior Full-stack Engineer                              | Armenia | Hybrid
  Product Marketing Manager, Krisp Voice AI               | Armenia | Hybrid


## Step 2 — Scrape detail pages

Each Krisp job detail page renders fully in plain HTML.  
Content is in `div.job_data_container` — title, location, work type, then structured sections.

In [4]:
records = []
for i, j in enumerate(armenia, 1):
    print(f"[{i}/{len(armenia)}] {j['url']}")
    r = requests.get(j["url"], headers=HEADERS, timeout=20)
    s = BeautifulSoup(r.text, "html.parser")
    container = s.find(class_="job_data_container")

    if not container:
        full_text = ""
    else:
        lines = container.get_text("\n", strip=True).split("\n")
        # Skip header lines (title, location, work_type already known)
        start = 0
        for idx, line in enumerate(lines):
            if line.strip() in (j["location"], j["work_type"]):
                start = idx + 1
                break
        full_text = "\n".join(lines[start:]).strip()

    records.append({
        "source":       "krisp",
        "source_url":   j["url"],
        "job_title":    j["title"],
        "company_name": "Krisp",
        "location":     j["location"],
        "work_type":    j["work_type"],
        "posting_date": "",
        "full_text":    full_text,
    })
    print(f"  full_text: {len(full_text)} chars")
    time.sleep(DELAY_S)

[1/7] https://krisp.ai/jobs/technical-support-engineering-manager-ai-meeting-assistant/
  full_text: 4270 chars
[2/7] https://krisp.ai/jobs/senior-customer-engineer-enterprise/
  full_text: 5364 chars
[3/7] https://krisp.ai/jobs/ai-qa-engineer/
  full_text: 2172 chars
[4/7] https://krisp.ai/jobs/lead-ml-systems-engineer-voice-ai/
  full_text: 4407 chars
[5/7] https://krisp.ai/jobs/ml-engineer-voice-ai/
  full_text: 4626 chars
[6/7] https://krisp.ai/jobs/senior-full-stack-engineer-3/
  full_text: 3727 chars
[7/7] https://krisp.ai/jobs/product-marketing-manager-krisp-voice-ai/
  full_text: 3261 chars


## Step 3 — Save outputs

In [5]:
df = pd.DataFrame(records)
raw_path = RAW_DIR / "krisp_jobs_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw saved: {raw_path} ({len(df)} rows)")

std_cols = ["source","source_url","job_title","company_name","location","work_type","posting_date","full_text"]
std_df = df[std_cols]
std_path = PROC_DIR / "krisp_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized saved: {std_path} ({len(std_df)} rows)")
print()
print("=== FIELD COVERAGE ===")
for col in std_cols:
    filled = std_df[col].replace("", pd.NA).notna().sum()
    print(f"  {col:20s}: {filled}/{len(std_df)} ({filled/len(std_df)*100:.0f}%)")
print()
ft = std_df["full_text"].str.len()
print(f"full_text — Min: {ft.min()}  Median: {ft.median():.0f}  Max: {ft.max()}")

Raw saved: /Users/lianaaghamalyan/thesis_data/data/raw/jobs/krisp_jobs_raw.csv (7 rows)
Standardized saved: /Users/lianaaghamalyan/thesis_data/data/processed/jobs/krisp_jobs_standardized.csv (7 rows)

=== FIELD COVERAGE ===
  source              : 7/7 (100%)
  source_url          : 7/7 (100%)
  job_title           : 7/7 (100%)
  company_name        : 7/7 (100%)
  location            : 7/7 (100%)
  work_type           : 7/7 (100%)
  posting_date        : 0/7 (0%)
  full_text           : 7/7 (100%)

full_text — Min: 2172  Median: 4270  Max: 5364

Note: posting_date not available on Krisp careers page (no dates displayed).
